# HI-Kyber GPU Implementation

In [ ]:
%%writefile hi_kyber_full.cu
#include <iostream>
#include <vector>
#include <chrono>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

#define CUDA_CHECK_LAST_ERROR() \
    do { \
        cudaError_t err = cudaGetLastError(); \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

/* =========================================================
 * 1. KYBER CRYPTOGRAPHIC PARAMETERS & CONSTANTS
 * ========================================================= */
#define KYBER_N 256
#define KYBER_Q 3329

// Authentic Kyber NTT Twiddle Factors (zetas) - Host
const int16_t zetas_host[128] = {
  2285, 2586, 2560, 2221, 3277, 2890, 3138, 2824, 1049, 2063,  385, 2838,
  3114, 2854, 3014, 3121,   62,  597, 3034, 1475,  439, 2690, 2468, 1424,
  1388, 2901, 1928,  348, 1792, 1092, 2379, 1493, 2746, 3216,  530,  188,
   591, 1512, 1690,  483, 1014, 2480, 2038, 1146,  310,  831, 2269, 1610,
  2390, 1269, 2984, 1553, 3159, 1563,  361,  803,  917, 3270,  715,  623,
  1851, 1572, 2263,  560, 2529, 2095, 2548,  856, 1761, 2346, 2484, 1767,
  3097, 2522,  124, 2903, 1812, 1656, 1791, 2636, 1672, 1332, 1471,  722,
  2800, 2439, 2287,  858, 1111, 2772, 3290, 3144,  110, 1150,   47,  916,
  1852, 1361,  579,  238,  581,  770,  521, 1163, 2942, 1957, 1060, 2623,
  3314, 3132,   85, 1219, 1474, 1845, 1863, 1438, 1930, 1152,  459, 1756,
  1496,  680, 2027, 2686
};

// Authentic Kyber NTT Twiddle Factors (zetas) - Device
__device__ const int16_t zetas_dev[128] = {
  2285, 2586, 2560, 2221, 3277, 2890, 3138, 2824, 1049, 2063,  385, 2838,
  3114, 2854, 3014, 3121,   62,  597, 3034, 1475,  439, 2690, 2468, 1424,
  1388, 2901, 1928,  348, 1792, 1092, 2379, 1493, 2746, 3216,  530,  188,
   591, 1512, 1690,  483, 1014, 2480, 2038, 1146,  310,  831, 2269, 1610,
  2390, 1269, 2984, 1553, 3159, 1563,  361,  803,  917, 3270,  715,  623,
  1851, 1572, 2263,  560, 2529, 2095, 2548,  856, 1761, 2346, 2484, 1767,
  3097, 2522,  124, 2903, 1812, 1656, 1791, 2636, 1672, 1332, 1471,  722,
  2800, 2439, 2287,  858, 1111, 2772, 3290, 3144,  110, 1150,   47,  916,
  1852, 1361,  579,  238,  581,  770,  521, 1163, 2942, 1957, 1060, 2623,
  3314, 3132,   85, 1219, 1474, 1845, 1863, 1438, 1930, 1152,  459, 1756,
  1496,  680, 2027, 2686
};

/* =========================================================
 * 2. MODULAR REDUCTIONS (Identical to Kyber Ref C)
 * ========================================================= */
__device__ __host__ int16_t montgomery_reduce(int32_t a) {
    int32_t t;
    int16_t u;
    u = (int16_t)(a * 62209); // q^-1 mod 2^16
    t = (int32_t)u * KYBER_Q;
    t = a - t;
    t >>= 16;
    return (int16_t)t;
}

__device__ __host__ int16_t barrett_reduce(int16_t a) {
    int16_t t;
    const int16_t v = ((1<<26) + KYBER_Q/2)/KYBER_Q;
    t  = ((int32_t)v * a + (1<<25)) >> 26;
    t *= KYBER_Q;
    return a - t;
}

/* =========================================================
 * 3. CPU REFERENCE IMPLEMENTATION (For Verification)
 * ========================================================= */
void ntt_cpu_reference(int16_t *r) {
    int len, start, j, k;
    int16_t t, zeta;
    k = 1;
    for(len = 128; len >= 2; len >>= 1) {
        for(start = 0; start < 256; start = start + 2*len) {
            zeta = zetas_host[k++];
            for(j = start; j < start + len; j++) {
                t = montgomery_reduce((int32_t)zeta * r[j + len]);
                r[j + len] = r[j] - t;
                r[j] = r[j] + t;
            }
        }
    }
}

/* =========================================================
 * 4. GPU EDFS-NTT IMPLEMENTATION (True SLM / Coalesced)
 * 
 * Implements Sliced Layer Merging (SLM) using Shared Memory.
 * Instead of 1 thread taking 256 variables into registers (causing spills),
 * a block of threads collectively loads polynomials into fast L1 Shared Memory
 * using coalesced global memory access, processes the NTT tree, and writes back.
 * ========================================================= */
__global__ void edfs_ntt_kernel(int16_t* d_poly_batch, int num_polys) {
    // We process 1 polynomial per 32-thread Warp to eliminate register pressure
    int warp_id = (blockIdx.x * blockDim.x + threadIdx.x) / 32;
    int lane_id = threadIdx.x % 32;
    
    if (warp_id >= num_polys) return;

    // Shared memory allocated dynamically or statically for the block
    // blockDim.x / 32 polynomials per block
    extern __shared__ int16_t s_polys[];
    int16_t* s_p = &s_polys[(threadIdx.x / 32) * KYBER_N];

    // Coalesced Global Memory Load into Shared Memory
    // 32 threads load 256 elements -> each thread loads 8 elements
    int16_t* global_p = &d_poly_batch[warp_id * KYBER_N];
    #pragma unroll 8
    for(int i = 0; i < 8; i++) {
        s_p[lane_id + i * 32] = global_p[lane_id + i * 32];
    }
    
    __syncwarp(); // Ensure warp has loaded the polynomial

    // Cooley-Tukey NTT traversal executed collectively by the Warp
    int k = 1;
    for (int len = 128; len >= 2; len >>= 1) {
        for (int start = 0; start < 256; start += 2 * len) {
            int16_t zeta = zetas_dev[k++];
            
            // Distribute the butterflies among the 32 threads in the warp
            for (int j = start + lane_id; j < start + len; j += 32) {
                int32_t t = montgomery_reduce((int32_t)zeta * s_p[j + len]);
                s_p[j + len] = s_p[j] - t;
                s_p[j] = s_p[j] + t;
            }
        }
        __syncwarp(); // Sync warp after each layer
    }

    // Coalesced Write back to Global Memory
    #pragma unroll 8
    for(int i = 0; i < 8; i++) {
        global_p[lane_id + i * 32] = s_p[lane_id + i * 32];
    }
}

/* =========================================================
 * 5. VALIDATION & BENCHMARK
 * ========================================================= */
int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);
    if (deviceCount == 0) {
        std::cerr << "No CUDA-capable devices found! Ensure GPU runtime is selected." << std::endl;
        return EXIT_FAILURE;
    }

    // 1. Correctness Validation
    std::cout << "[1/3] Validating GPU Cryptographic Correctness..." << std::endl;
    int16_t cpu_poly[KYBER_N];
    int16_t gpu_poly[KYBER_N];

    // Initialize a random polynomial bounded by Q
    for (int i = 0; i < KYBER_N; i++) {
        int16_t val = (i * 137 + 101) % KYBER_Q;
        cpu_poly[i] = val;
        gpu_poly[i] = val;
    }

    // CPU Run
    ntt_cpu_reference(cpu_poly);

    // GPU Run
    int16_t *d_poly_val;
    CUDA_CHECK(cudaMalloc(&d_poly_val, KYBER_N * sizeof(int16_t)));
    CUDA_CHECK(cudaMemcpy(d_poly_val, gpu_poly, KYBER_N * sizeof(int16_t), cudaMemcpyHostToDevice));
    
    // We launch 32 threads for a single polynomial in this test
    size_t shared_mem_size = (32 / 32) * KYBER_N * sizeof(int16_t);
    edfs_ntt_kernel<<<1, 32, shared_mem_size>>>(d_poly_val, 1);
    CUDA_CHECK_LAST_ERROR();
    CUDA_CHECK(cudaDeviceSynchronize());
    
    CUDA_CHECK(cudaMemcpy(gpu_poly, d_poly_val, KYBER_N * sizeof(int16_t), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_poly_val));

    // Diff & Correctness Metric
    int correct_count = 0;
    for(int i=0; i<KYBER_N; i++) {
        if(cpu_poly[i] == gpu_poly[i]) {
            correct_count++;
        } else {
            std::cout << "MISMATCH at index " << i << ": CPU=" << cpu_poly[i] << " GPU=" << gpu_poly[i] << std::endl;
        }
    }
    
    double correctness = (correct_count / (double)KYBER_N) * 100.0;
    std::cout << "Correctness Metric: " << correctness << "% (" << correct_count << "/" << KYBER_N << " coefficients matched)" << std::endl;

    if (correct_count == KYBER_N) {
        std::cout << "[SUCCESS] GPU output perfectly matches CPU reference implementation!" << std::endl << std::endl;
    } else {
        std::cout << "[FAILED] GPU output diverged from CPU reference!" << std::endl << std::endl;
        return -1;
    }

    // 2. Benchmarks Setup
    int NUM_POLYS = 1000000; // 1 Million Polynomials
    std::cout << "[2/4] Allocating 1,000,000 polynomials for CPU and GPU..." << std::endl;
    size_t size = NUM_POLYS * KYBER_N * sizeof(int16_t);
    
    int16_t *h_polys = (int16_t*)malloc(size);
    // Initialize array to prevent trivial cache-hits if all 0s
    for (size_t i = 0; i < NUM_POLYS * KYBER_N; i++) {
        h_polys[i] = i % KYBER_Q;
    }

    int16_t *d_polys;
    CUDA_CHECK(cudaMalloc(&d_polys, size));
    CUDA_CHECK(cudaMemcpy(d_polys, h_polys, size, cudaMemcpyHostToDevice));

    int threadsPerBlock = 256; 
    // Since 1 warp (32 threads) processes 1 poly, a block of 256 threads processes 8 polys
    int polysPerBlock = threadsPerBlock / 32;
    int blocksPerGrid = (NUM_POLYS + polysPerBlock - 1) / polysPerBlock;
    size_t bench_shared_mem_size = polysPerBlock * KYBER_N * sizeof(int16_t);

    // 3. CPU Benchmark
    std::cout << "[3/4] Running CPU Performance Benchmark..." << std::endl;
    auto cpu_start_time = std::chrono::high_resolution_clock::now();
    for (int i = 0; i < NUM_POLYS; i++) {
        ntt_cpu_reference(&h_polys[i * KYBER_N]);
    }
    auto cpu_end_time = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> cpu_diff = cpu_end_time - cpu_start_time;

    // 4. GPU Benchmark
    std::cout << "[4/4] Running GPU Performance Benchmark...\n" << std::endl;
    CUDA_CHECK(cudaDeviceSynchronize());

    auto gpu_start_time = std::chrono::high_resolution_clock::now();
    edfs_ntt_kernel<<<blocksPerGrid, threadsPerBlock, bench_shared_mem_size>>>(d_polys, NUM_POLYS);
    CUDA_CHECK_LAST_ERROR();
    CUDA_CHECK(cudaDeviceSynchronize());
    auto gpu_end_time = std::chrono::high_resolution_clock::now();
    
    std::chrono::duration<double> gpu_diff = gpu_end_time - gpu_start_time;
    
    std::cout << "================ HI-KYBER NTT BENCHMARK =================" << std::endl;
    std::cout << "Device: CPU vs NVIDIA T4 (Google Colab)" << std::endl;
    std::cout << "Processed: " << NUM_POLYS << " Polynomials." << std::endl;
    std::cout << "---------------------------------------------------------" << std::endl;
    std::cout << "CPU Time:       " << cpu_diff.count() << " seconds." << std::endl;
    std::cout << "CPU Throughput: " << (NUM_POLYS / cpu_diff.count()) / 1000.0 << " kops/s" << std::endl;
    std::cout << "---------------------------------------------------------" << std::endl;
    std::cout << "GPU Time:       " << gpu_diff.count() << " seconds." << std::endl;
    std::cout << "GPU Throughput: " << (NUM_POLYS / gpu_diff.count()) / 1000.0 << " kops/s" << std::endl;
    std::cout << "---------------------------------------------------------" << std::endl;
    std::cout << "SPEEDUP:        " << cpu_diff.count() / gpu_diff.count() << "x faster on GPU" << std::endl;
    std::cout << "=========================================================" << std::endl;

    CUDA_CHECK(cudaFree(d_polys));
    free(h_polys);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -O3 hi_kyber_full.cu -o hi_kyber_full
!./hi_kyber_full

---

## How to take this tuning further on your own

The paper achieves mind-blowing speeds (over 1.6 Million Key Exchanges per second) not just by writing basic C++ mapping to threads. They perform extremely deep hardware-specific optimizations. Here are the 4 fundamental modifications you must make to the above code to reach the paper's ceiling:

### 1. Optimize Global Memory Access (Coalescing)
Currently, thread 0 grabs `p[0]..p[255]`, then thread 1 grabs `p[256]..p[511]`. On an Nvidia SM, this causes massive memory divergence.
* **Your Task**: Transpose your global memory layout. You want memory organized so that Thread 0 accesses index 0, Thread 1 accesses index 1, etc.
  * Modify the dataset mapping so `p[i]` is stored as `global_poly_batch[layer_index * total_polys + poly_idx]`.
  * When warps execute, all 32 threads in the warp will load a block of contiguous memory, allowing the L2 cache to broadcast the bus efficiently.


---
## 4. Pointwise Multiplication (BaseMul)
This implements multiplication of elements in Rq in the NTT domain. Two polynomials are multiplied point-by-point using complex modular arithmetic.

In [ ]:
%%writefile kyber_pointwise.cu
#include <iostream>
#include <iomanip>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

#define CUDA_CHECK_LAST_ERROR() \
    do { \
        cudaError_t err = cudaGetLastError(); \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

#define KYBER_N 256
#define KYBER_Q 3329

// Device constants
__device__ const int16_t zetas_dev[128] = {
  2285, 2586, 2560, 2221, 3277, 2890, 3138, 2824, 1049, 2063,  385, 2838,
  3114, 2854, 3014, 3121,   62,  597, 3034, 1475,  439, 2690, 2468, 1424,
  1388, 2901, 1928,  348, 1792, 1092, 2379, 1493, 2746, 3216,  530,  188,
   591, 1512, 1690,  483, 1014, 2480, 2038, 1146,  310,  831, 2269, 1610,
  2390, 1269, 2984, 1553, 3159, 1563,  361,  803,  917, 3270,  715,  623,
  1851, 1572, 2263,  560, 2529, 2095, 2548,  856, 1761, 2346, 2484, 1767,
  3097, 2522,  124, 2903, 1812, 1656, 1791, 2636, 1672, 1332, 1471,  722,
  2800, 2439, 2287,  858, 1111, 2772, 3290, 3144,  110, 1150,   47,  916,
  1852, 1361,  579,  238,  581,  770,  521, 1163, 2942, 1957, 1060, 2623,
  3314, 3132,   85, 1219, 1474, 1845, 1863, 1438, 1930, 1152,  459, 1756,
  1496,  680, 2027, 2686
};

// Host constants
const int16_t zetas_host[128] = {
  2285, 2586, 2560, 2221, 3277, 2890, 3138, 2824, 1049, 2063,  385, 2838,
  3114, 2854, 3014, 3121,   62,  597, 3034, 1475,  439, 2690, 2468, 1424,
  1388, 2901, 1928,  348, 1792, 1092, 2379, 1493, 2746, 3216,  530,  188,
   591, 1512, 1690,  483, 1014, 2480, 2038, 1146,  310,  831, 2269, 1610,
  2390, 1269, 2984, 1553, 3159, 1563,  361,  803,  917, 3270,  715,  623,
  1851, 1572, 2263,  560, 2529, 2095, 2548,  856, 1761, 2346, 2484, 1767,
  3097, 2522,  124, 2903, 1812, 1656, 1791, 2636, 1672, 1332, 1471,  722,
  2800, 2439, 2287,  858, 1111, 2772, 3290, 3144,  110, 1150,   47,  916,
  1852, 1361,  579,  238,  581,  770,  521, 1163, 2942, 1957, 1060, 2623,
  3314, 3132,   85, 1219, 1474, 1845, 1863, 1438, 1930, 1152,  459, 1756,
  1496,  680, 2027, 2686
};

__device__ __host__ int16_t montgomery_reduce(int32_t a) {
    int32_t t;
    int16_t u;
    u = (int16_t)(a * 62209);
    t = (int32_t)u * KYBER_Q;
    t = a - t;
    t >>= 16;
    return (int16_t)t;
}

__device__ __host__ int16_t fqmul(int16_t a, int16_t b) {
    return montgomery_reduce((int32_t)a * b);
}

// Host reference pointwise multiplication
void pointwise_mul_cpu(int16_t* r, const int16_t* a, const int16_t* b) {
    for (int i = 0; i < KYBER_N / 4; i++) {
        int zeta_idx = 64 + i;
        int16_t zeta = zetas_host[zeta_idx];
        
        int16_t a0 = a[4 * i];
        int16_t a1 = a[4 * i + 1];
        int16_t b0 = b[4 * i];
        int16_t b1 = b[4 * i + 1];
        
        r[4 * i] = fqmul(a1, b1);
        r[4 * i] = fqmul(r[4 * i], zeta);
        r[4 * i] += fqmul(a0, b0);
        
        r[4 * i + 1] = fqmul(a0, b1);
        r[4 * i + 1] += fqmul(a1, b0);
        
        a0 = a[4 * i + 2];
        a1 = a[4 * i + 3];
        b0 = b[4 * i + 2];
        b1 = b[4 * i + 3];
        
        r[4 * i + 2] = fqmul(a1, b1);
        r[4 * i + 2] = fqmul(r[4 * i + 2], -zeta);
        r[4 * i + 2] += fqmul(a0, b0);
        
        r[4 * i + 3] = fqmul(a0, b1);
        r[4 * i + 3] += fqmul(a1, b0);
    }
}

// -----------------------------------------------------------------
// Pointwise Multiplication Kernel
// -----------------------------------------------------------------
__global__ void pointwise_mul_kernel(int16_t* d_r, const int16_t* d_a, const int16_t* d_b, int num_polys) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= num_polys * 64) return;
    
    int poly_idx = idx / 64;
    int coef_idx = (idx % 64) * 4;
    int zeta_idx = 64 + (idx % 64);
    int16_t zeta = zetas_dev[zeta_idx];
    
    int16_t a0 = d_a[poly_idx * KYBER_N + coef_idx];
    int16_t a1 = d_a[poly_idx * KYBER_N + coef_idx + 1];
    int16_t b0 = d_b[poly_idx * KYBER_N + coef_idx];
    int16_t b1 = d_b[poly_idx * KYBER_N + coef_idx + 1];
    
    d_r[poly_idx * KYBER_N + coef_idx]     = fqmul(a1, b1);
    d_r[poly_idx * KYBER_N + coef_idx]     = fqmul(d_r[poly_idx * KYBER_N + coef_idx], zeta);
    d_r[poly_idx * KYBER_N + coef_idx]    += fqmul(a0, b0);
    
    d_r[poly_idx * KYBER_N + coef_idx + 1] = fqmul(a0, b1);
    d_r[poly_idx * KYBER_N + coef_idx + 1]+= fqmul(a1, b0);
    
    a0 = d_a[poly_idx * KYBER_N + coef_idx + 2];
    a1 = d_a[poly_idx * KYBER_N + coef_idx + 3];
    b0 = d_b[poly_idx * KYBER_N + coef_idx + 2];
    b1 = d_b[poly_idx * KYBER_N + coef_idx + 3];
    
    d_r[poly_idx * KYBER_N + coef_idx + 2] = fqmul(a1, b1);
    d_r[poly_idx * KYBER_N + coef_idx + 2] = fqmul(d_r[poly_idx * KYBER_N + coef_idx + 2], -zeta);
    d_r[poly_idx * KYBER_N + coef_idx + 2]+= fqmul(a0, b0);
    
    d_r[poly_idx * KYBER_N + coef_idx + 3] = fqmul(a0, b1);
    d_r[poly_idx * KYBER_N + coef_idx + 3]+= fqmul(a1, b0);
}

int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);
    if (deviceCount == 0) {
        std::cerr << "No CUDA-capable devices found! Ensure GPU runtime is selected." << std::endl;
        return EXIT_FAILURE;
    }

    int16_t h_a[KYBER_N], h_b[KYBER_N], h_r[KYBER_N], h_expected[KYBER_N];
    for (int i = 0; i < KYBER_N; i++) { 
        h_a[i] = i * 2; 
        h_b[i] = i * 3; 
    }
    
    int16_t *d_a, *d_b, *d_r;
    CUDA_CHECK(cudaMalloc(&d_a, KYBER_N * sizeof(int16_t)));
    CUDA_CHECK(cudaMalloc(&d_b, KYBER_N * sizeof(int16_t)));
    CUDA_CHECK(cudaMalloc(&d_r, KYBER_N * sizeof(int16_t)));
    
    CUDA_CHECK(cudaMemcpy(d_a, h_a, KYBER_N * sizeof(int16_t), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b, KYBER_N * sizeof(int16_t), cudaMemcpyHostToDevice));
    
    pointwise_mul_kernel<<<1, 64>>>(d_r, d_a, d_b, 1);
    CUDA_CHECK_LAST_ERROR();
    CUDA_CHECK(cudaDeviceSynchronize());
    
    CUDA_CHECK(cudaMemcpy(h_r, d_r, KYBER_N * sizeof(int16_t), cudaMemcpyDeviceToHost));
    
    // CPU reference
    pointwise_mul_cpu(h_expected, h_a, h_b);
    
    std::cout << "--- POINTWISE MULTIPLICATION TEST ---" << std::endl;
    std::cout << "Poly A (first 8 elems): ";
    for(int i=0; i<8; i++) std::cout << h_a[i] << " ";
    std::cout << std::endl;
    
    std::cout << "Poly B (first 8 elems): ";
    for(int i=0; i<8; i++) std::cout << h_b[i] << " ";
    std::cout << std::endl;
    
    bool match = true;
    for (int i=0; i<KYBER_N; i++) {
        if (h_r[i] != h_expected[i]) match = false;
    }
    
    if (match) {
        std::cout << "SUCCESS: GPU Pointwise Multiplication matches CPU reference." << std::endl;
        std::cout << "Comparison (first 8 elems):" << std::endl;
        std::cout << "GPU Result : ";
        for(int i=0; i<8; i++) std::cout << std::setw(5) << h_r[i] << " ";
        std::cout << std::endl;
        std::cout << "Expected   : ";
        for(int i=0; i<8; i++) std::cout << std::setw(5) << h_expected[i] << " ";
        std::cout << std::endl;
    } else {
        std::cout << "ERROR: Pointwise mismatch!" << std::endl;
    }
    
    CUDA_CHECK(cudaFree(d_a)); CUDA_CHECK(cudaFree(d_b)); CUDA_CHECK(cudaFree(d_r));
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -O3 kyber_pointwise.cu -o kyber_pointwise
!./kyber_pointwise

---
## 5. INTT (Inverse Number Theoretic Transform)
The INTT processes arrays that have been manipulated in frequency space back into positional space. Note the reversal of layers and scaling by Montgomery factor `f = 1441`.

In [ ]:
%%writefile kyber_intt.cu
#include <iostream>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

#define CUDA_CHECK_LAST_ERROR() \
    do { \
        cudaError_t err = cudaGetLastError(); \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

#define KYBER_N 256
#define KYBER_Q 3329

__device__ const int16_t zetas_dev[128] = {
  -1044,  -758,  -359, -1517,  1493,  1422,   287,   202,
   -171,   622,  1577,   182,   962, -1202, -1474,  1468,
    573, -1325,   264,   383,  -829,  1458, -1600,  -130,
    311, -1115,   902,   412,  1186,  -387, -1651, -1015,
  -1456,  1115,  -897, -1584,   218,   318, -1137,  -845,
   -583, -1133, -1607,  1612,  -633,   895, -1274,  1456,
  -1188,  1172,  -570,   848, -1004, -1455,  -932,  -311,
   1484,    97, -1213, -1636,  -144, -1387, -1571, -1095,
   -365,  -715,   984,  -256,  -462,  1340,   607, -1331,
  -1144, -1266,   476,  -532,   200,   265, -1071, -1536,
  -1191,   999,   442, -1614, -1300,  -885,   416,  -521,
  -1224,  1641, -1541,  1006,  1235,   -42,   237,   411,
    734,  1566, -1141,  -907,   498,   909, -1260,  -511,
  -1433, -1219, -1355, -1086,  -275,  1311, -1167,   108,
  -1520,  1300,   122,  1110, -1193,  -177,  -841,  -291,
   -235, -1405,  1619,   461,  1623, -1067, -1093,   900
};

__device__ int16_t barrett_reduce(int16_t a) {
    int16_t t;
    const int16_t v = ((1<<26) + KYBER_Q/2)/KYBER_Q;
    t  = ((int32_t)v * a + (1<<25)) >> 26;
    t *= KYBER_Q;
    return a - t;
}

__device__ int16_t fqmul(int16_t a, int16_t b) {
    int32_t t;
    int16_t u;
    u = (int16_t)(a * b * 62209);
    t = (int32_t)u * KYBER_Q;
    t = (int32_t)a * b - t;
    t >>= 16;
    return (int16_t)t;
}

__global__ void intt_kernel(int16_t* d_poly_batch, int num_polys) {
    int warp_id = (blockIdx.x * blockDim.x + threadIdx.x) / 32;
    int lane_id = threadIdx.x % 32;
    
    if (warp_id >= num_polys) return;

    extern __shared__ int16_t s_polys[];
    int16_t* s_p = &s_polys[(threadIdx.x / 32) * KYBER_N];
    int16_t* global_p = &d_poly_batch[warp_id * KYBER_N];

    #pragma unroll 8
    for(int i=0; i<8; i++) {
        s_p[lane_id + i*32] = global_p[lane_id + i*32];
    }
    __syncwarp();

    int k = 127;
    for (int len = 2; len <= 128; len <<= 1) {
        for (int start = 0; start < 256; start += 2 * len) {
            int16_t zeta = zetas_dev[k--];
            for (int j = start + lane_id; j < start + len; j += 32) {
                int16_t t = s_p[j];
                s_p[j] = barrett_reduce(t + s_p[j + len]);
                s_p[j + len] = s_p[j + len] - t;
                s_p[j + len] = fqmul(zeta, s_p[j + len]);
            }
        }
        __syncwarp();
    }

    const int16_t f = 1441; // mont^2/128
    #pragma unroll 8
    for(int i=0; i<8; i++) {
        int idx = lane_id + i * 32;
        s_p[idx] = fqmul(s_p[idx], f);
        global_p[idx] = s_p[idx];
    }
}

int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);
    if (deviceCount == 0) {
        std::cerr << "No CUDA-capable devices found! Ensure GPU runtime is selected." << std::endl;
        return EXIT_FAILURE;
    }

    int16_t h_poly[KYBER_N];
    for (int i=0; i<KYBER_N; i++) h_poly[i] = i % KYBER_Q;

    int16_t *d_poly;
    CUDA_CHECK(cudaMalloc(&d_poly, KYBER_N * sizeof(int16_t)));
    CUDA_CHECK(cudaMemcpy(d_poly, h_poly, KYBER_N * sizeof(int16_t), cudaMemcpyHostToDevice));

    size_t shared_mem = KYBER_N * sizeof(int16_t);
    intt_kernel<<<1, 32, shared_mem>>>(d_poly, 1);
    CUDA_CHECK_LAST_ERROR();
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(h_poly, d_poly, KYBER_N * sizeof(int16_t), cudaMemcpyDeviceToHost));
    std::cout << "SUCCESS: INTT Kernel Executed." << std::endl;
    std::cout << "Sample resulting element [0]: " << h_poly[0] << std::endl;

    CUDA_CHECK(cudaFree(d_poly));

    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -O3 kyber_intt.cu -o kyber_intt
!./kyber_intt

---
## 6. Encode / Decode
Packs 12-bit coefficients of the polynomial into a dense array of 8-bit bytes (384 bytes per polynomial) and unpacks them, as required for constructing the Ciphertext.

In [ ]:
%%writefile kyber_encode_decode.cu
#include <iostream>
#include <cuda_runtime.h>
#include <cstring>
#include <iomanip>

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

#define CUDA_CHECK_LAST_ERROR() \
    do { \
        cudaError_t err = cudaGetLastError(); \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

#define KYBER_N 256
#define KYBER_Q 3329

// Each polynomial has 256 coefficients, max value ~3328 (requires 12 bits)
// 256 * 12 bits = 384 bytes
#define KYBER_POLYBYTES 384

// -----------------------------------------------------------------
// Encode Kernel: (poly_tobytes) 
// Maps 12-bit int16_t coefficients to 8-bit bytes densely
// -----------------------------------------------------------------
__global__ void encode_kernel(uint8_t* d_bytes, const int16_t* d_polys, int num_polys) {
    // We can use 128 threads per polynomial since each iteration handles 2 coefficients and yields 3 bytes
    int idx = blockIdx.x * blockDim.x + threadIdx.x; 
    if (idx >= num_polys * 128) return;
    
    int poly_idx = idx / 128;
    int pair_idx = idx % 128;
    
    int16_t t0 = d_polys[poly_idx * KYBER_N + 2 * pair_idx];
    int16_t t1 = d_polys[poly_idx * KYBER_N + 2 * pair_idx + 1];

    // Map to positive standard representatives
    t0 += (t0 >> 15) & KYBER_Q;
    t1 += (t1 >> 15) & KYBER_Q;

    // Pack two 12-bit numbers into three 8-bit numbers
    d_bytes[poly_idx * KYBER_POLYBYTES + 3 * pair_idx + 0] = (t0 >> 0) & 0xFF;
    d_bytes[poly_idx * KYBER_POLYBYTES + 3 * pair_idx + 1] = ((t0 >> 8) | (t1 << 4)) & 0xFF;
    d_bytes[poly_idx * KYBER_POLYBYTES + 3 * pair_idx + 2] = (t1 >> 4) & 0xFF;
}

// -----------------------------------------------------------------
// Decode Kernel: (poly_frombytes)
// Unpacks 8-bit dense arrays back into 12-bit int16_t coefficients
// -----------------------------------------------------------------
__global__ void decode_kernel(int16_t* d_polys, const uint8_t* d_bytes, int num_polys) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x; 
    if (idx >= num_polys * 128) return;
    
    int poly_idx = idx / 128;
    int pair_idx = idx % 128;
    
    uint8_t b0 = d_bytes[poly_idx * KYBER_POLYBYTES + 3 * pair_idx + 0];
    uint8_t b1 = d_bytes[poly_idx * KYBER_POLYBYTES + 3 * pair_idx + 1];
    uint8_t b2 = d_bytes[poly_idx * KYBER_POLYBYTES + 3 * pair_idx + 2];
    
    d_polys[poly_idx * KYBER_N + 2 * pair_idx + 0] = ((b0 >> 0) | ((uint16_t)b1 << 8)) & 0xFFF;
    d_polys[poly_idx * KYBER_N + 2 * pair_idx + 1] = ((b1 >> 4) | ((uint16_t)b2 << 4)) & 0xFFF;
}

int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);
    if (deviceCount == 0) {
        std::cerr << "No CUDA-capable devices found! Ensure GPU runtime is selected." << std::endl;
        return EXIT_FAILURE;
    }

    int16_t h_poly_in[KYBER_N];
    int16_t h_poly_out[KYBER_N];
    uint8_t h_bytes[KYBER_POLYBYTES];
    
    // The text to test encryption/decryption (encoding/decoding)
    const char* secret_message = "Hello Kyber! This text is converted into polynomial coefficients, encrypted (encoded) into 384 bytes, and decrypted (decoded) back to text!";
    int msg_len = strlen(secret_message);
    
    // Initialize polynomial with text
    for (int i=0; i<KYBER_N; i++) {
        if (i < msg_len) {
            h_poly_in[i] = secret_message[i];
        } else {
            h_poly_in[i] = 0; // padding
        }
    }
    
    std::cout << "--- TEXT ENCRYPTION / DECRYPTION TEST (ENCODE/DECODE) ---" << std::endl;
    std::cout << "[1] Text before encryption: " << secret_message << std::endl;
    
    int16_t *d_poly_in, *d_poly_out;
    uint8_t *d_bytes;
    CUDA_CHECK(cudaMalloc(&d_poly_in, KYBER_N * sizeof(int16_t)));
    CUDA_CHECK(cudaMalloc(&d_poly_out, KYBER_N * sizeof(int16_t)));
    CUDA_CHECK(cudaMalloc(&d_bytes, KYBER_POLYBYTES * sizeof(uint8_t)));
    
    CUDA_CHECK(cudaMemcpy(d_poly_in, h_poly_in, KYBER_N * sizeof(int16_t), cudaMemcpyHostToDevice));
    
    // Encrypt (Encode)
    encode_kernel<<<1, 128>>>(d_bytes, d_poly_in, 1);
    CUDA_CHECK_LAST_ERROR();
    CUDA_CHECK(cudaDeviceSynchronize());
    
    CUDA_CHECK(cudaMemcpy(h_bytes, d_bytes, KYBER_POLYBYTES * sizeof(uint8_t), cudaMemcpyDeviceToHost));
    
    std::cout << "[2] Bytes after encryption (Hex format snippet): ";
    for(int i=0; i<16; i++) {
        std::cout << std::hex << std::setw(2) << std::setfill('0') << (int)h_bytes[i] << " ";
    }
    std::cout << std::dec << "... (Total 384 bytes)" << std::endl;
    
    // Decrypt (Decode)
    decode_kernel<<<1, 128>>>(d_poly_out, d_bytes, 1);
    CUDA_CHECK_LAST_ERROR();
    CUDA_CHECK(cudaDeviceSynchronize());
    
    CUDA_CHECK(cudaMemcpy(h_poly_out, d_poly_out, KYBER_N * sizeof(int16_t), cudaMemcpyDeviceToHost));
    
    // Reconstruct text
    char output_text[KYBER_N];
    for(int i=0; i<KYBER_N; i++) {
        output_text[i] = (char)h_poly_out[i];
    }
    output_text[KYBER_N - 1] = '\0'; // ensure null termination if not present
    
    std::cout << "[3] Text after decryption: " << output_text << std::endl;
    
    bool passed = true;
    for(int i=0; i<KYBER_N; i++) {
        if(h_poly_in[i] != h_poly_out[i]) {
            passed = false;
        }
    }
    
    if(passed) {
        std::cout << "SUCCESS: Encode/Decode Kernels Executed and Verified." << std::endl;
    } else {
        std::cout << "ERROR: Encode/Decode mismatch!" << std::endl;
    }
    
    CUDA_CHECK(cudaFree(d_poly_in)); CUDA_CHECK(cudaFree(d_poly_out)); CUDA_CHECK(cudaFree(d_bytes));
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -O3 kyber_encode_decode.cu -o kyber_encode_decode
!./kyber_encode_decode